# 03. TP + timeout + downside multi-task MPTSNet

TP-only BCE가 큰 상승과 큰 하락을 함께 고득점으로 주는 문제를 수정한다.

- `TP head`: 미래 완성봉 `min_bid`가 비용 차감 후 +3%를 유지할 확률
- `timeout head`: 진입 5분 후 실제 순수익률의 조건부 평균
- `downside head`: 미래 1~5분 `min_bid` 순수익률 최솟값의 조건부 20% quantile
- TP class weight는 각 Train fold에서만 계산한다.
- 진입 utility는 TP 기대수익과 timeout 기대수익에서 예상 downside penalty를 차감한다.
- 최종 Test 날짜는 모델·penalty·threshold 선택에 사용하지 않는다.

현재 notebook은 **진입 AI**다. 매도 AI는 진입 후 매분 `hold/exit` 표본을 별도로 만든 뒤 다음 단계에서 추가한다.

In [1]:
from pathlib import Path
import gc
import json
import math
import random
import time

import nbformat
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, brier_score_loss, mean_absolute_error, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, Dataset


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
base_notebook = nbformat.read(PROJECT_ROOT / 'notebooks/02_compact_mptsnet_weighted_oof.ipynb', as_version=4)
for cell_index, marker in [(1, 'PREPROCESS_VERSION'), (3, 'fit_robust_scaler'), (5, 'CompactMPTSNet')]:
    source = base_notebook.cells[cell_index].source
    assert marker in source, (cell_index, marker)
    exec(compile(source, f'02_base_cell_{cell_index}', 'exec'), globals())

MODEL_VERSION = 'multitask_mptsnet_tp_timeout_downside_v1'
BATCH_SIZE = 512
MAX_EPOCHS = 30
PATIENCE = 6
TIMEOUT_LOSS_WEIGHT = 0.25
DOWNSIDE_LOSS_WEIGHT = 0.25
DOWNSIDE_QUANTILE = 0.20
MIN_OOF_TRADES = 30
RESULT_ROOT = (PROJECT_ROOT / '../../results/modeling').resolve()
MODEL_ROOT = (PROJECT_ROOT / '../../model').resolve()
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

future_min_return_columns = [f'net_return_min_bid_{minute}' for minute in range(1, 6)]
timeout_return = metadata['timeout_net_return_5m'].to_numpy(np.float32)
future_downside = metadata[future_min_return_columns].min(axis=1).to_numpy(np.float32)
trade_return = np.where(y == 1, 0.03, timeout_return).astype(np.float32)
timeout_target_pct = np.clip(timeout_return * 100, -25, 25).astype(np.float32)
downside_target_pct = np.clip(future_downside * 100, -30, 10).astype(np.float32)

print('model version:', MODEL_VERSION)
print('timeout return percentiles:', np.percentile(timeout_return, [1, 50, 99]))
print('downside percentiles:', np.percentile(future_downside, [1, 50, 99]))

device: cuda
X: (72181, 30, 38) | positive: 3.94%
OOF validation: ['session_2026-07-14', 'session_2026-07-15', 'session_2026-07-16']
final Test: session_2026-07-17
model version: multitask_mptsnet_tp_timeout_downside_v1
timeout return percentiles: [-0.09101166 -0.00600253  0.086385  ]
downside percentiles: [-0.11480958 -0.01460211 -0.00099062]


## Multi-task head와 loss

공유 MPTSNet 표현 위에 3개 head를 둔다. return target은 percentage point 단위로 학습해 BCE와 회귀 loss의 크기를 맞춘다. downside는 평균이 아니라 20% quantile pinball loss를 사용한다.

In [2]:
class MultiTaskDataset(Dataset):
    def __init__(self, x, indices):
        self.x = x
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        index = self.indices[item]
        return (
            torch.from_numpy(self.x[index]), torch.tensor(y[index]),
            torch.tensor(timeout_target_pct[index]), torch.tensor(downside_target_pct[index]), index,
        )


class MultiTaskMPTSNet(nn.Module):
    def __init__(self, input_dim, seq_len, periods, d_model=64, n_heads=4, blocks=2, dropout=0.15):
        super().__init__()
        encoder = CompactMPTSNet(input_dim, seq_len, periods, d_model, n_heads, blocks, dropout)
        self.input_projection = encoder.input_projection
        self.position = encoder.position
        self.blocks = encoder.blocks
        self.shared = nn.Sequential(
            nn.LayerNorm(d_model * 3), nn.Linear(d_model * 3, d_model), nn.GELU(), nn.Dropout(dropout),
        )
        self.tp_head = nn.Linear(d_model, 1)
        self.timeout_head = nn.Linear(d_model, 1)
        self.downside_head = nn.Linear(d_model, 1)

    def forward(self, x):
        x = self.input_projection(x) + self.position
        for block in self.blocks:
            x = block(x)
        pooled = torch.cat([x.mean(dim=1), x.amax(dim=1), x[:, -1]], dim=1)
        shared = self.shared(pooled)
        return {
            'tp_logit': self.tp_head(shared).squeeze(1),
            'timeout_pct': self.timeout_head(shared).squeeze(1),
            'downside_pct': self.downside_head(shared).squeeze(1),
        }


def pinball_loss(prediction, target, quantile):
    error = target - prediction
    return torch.maximum(quantile * error, (quantile - 1) * error).mean()


def spearman(actual, predicted):
    return float(pd.Series(actual).corr(pd.Series(predicted), method='spearman'))


def predicted_utility(tp_probability, predicted_timeout, predicted_downside, downside_penalty=0.0):
    base = tp_probability * 0.03 + (1 - tp_probability) * predicted_timeout
    return base - downside_penalty * np.maximum(-predicted_downside, 0)

## Expanding OOF 학습

TP PR-AUC와 직접 trade return에 대한 utility 순위 상관을 함께 사용해 early stopping한다. scaler, FFT period, class weight는 매 fold의 과거 날짜에서만 계산한다.

In [3]:
@torch.no_grad()
def predict_multitask(model, x_scaled, indices):
    loader = DataLoader(MultiTaskDataset(x_scaled, indices), batch_size=BATCH_SIZE, shuffle=False)
    model.eval()
    result = {'index': [], 'tp_probability': [], 'predicted_timeout': [], 'predicted_downside': []}
    for batch_x, _, _, _, batch_indices in loader:
        output = model(batch_x.to(DEVICE, non_blocking=True))
        result['index'].append(batch_indices.numpy())
        result['tp_probability'].append(torch.sigmoid(output['tp_logit']).cpu().numpy())
        result['predicted_timeout'].append(output['timeout_pct'].cpu().numpy() / 100)
        result['predicted_downside'].append(output['downside_pct'].cpu().numpy() / 100)
    return {key: np.concatenate(value) for key, value in result.items()}


def multitask_metrics(prediction):
    index = prediction['index']
    probability = prediction['tp_probability']
    utility = predicted_utility(probability, prediction['predicted_timeout'], prediction['predicted_downside'])
    return {
        'samples': int(len(index)), 'positives': int(y[index].sum()), 'prevalence': float(y[index].mean()),
        'pr_auc': float(average_precision_score(y[index], probability)),
        'roc_auc': float(roc_auc_score(y[index], probability)),
        'brier': float(brier_score_loss(y[index], probability)),
        'timeout_mae': float(mean_absolute_error(timeout_return[index], prediction['predicted_timeout'])),
        'timeout_spearman': spearman(timeout_return[index], prediction['predicted_timeout']),
        'downside_mae': float(mean_absolute_error(future_downside[index], prediction['predicted_downside'])),
        'downside_spearman': spearman(future_downside[index], prediction['predicted_downside']),
        'utility_spearman': spearman(trade_return[index], utility),
    }


def fit_fold_multitask(x_scaled, train_indices, valid_indices, periods, fold_seed):
    torch.manual_seed(fold_seed)
    model = MultiTaskMPTSNet(x_scaled.shape[-1], x_scaled.shape[1], periods).to(DEVICE)
    train_y = y[train_indices]
    pos_weight_value = float((len(train_y) - train_y.sum()) / max(train_y.sum(), 1))
    tp_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, device=DEVICE))
    timeout_criterion = nn.HuberLoss(delta=1.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=3e-5)
    generator = torch.Generator().manual_seed(fold_seed)
    loader = DataLoader(
        MultiTaskDataset(x_scaled, train_indices), batch_size=BATCH_SIZE, shuffle=True,
        generator=generator, pin_memory=DEVICE.type == 'cuda',
    )
    best_score, best_state, best_epoch, stale = -np.inf, None, 0, 0
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        totals = {'loss': 0.0, 'tp': 0.0, 'timeout': 0.0, 'downside': 0.0}
        for batch_x, batch_y, batch_timeout, batch_downside, _ in loader:
            batch_x = batch_x.to(DEVICE, non_blocking=True)
            batch_y = batch_y.to(DEVICE, non_blocking=True)
            batch_timeout = batch_timeout.to(DEVICE, non_blocking=True)
            batch_downside = batch_downside.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            output = model(batch_x)
            tp_loss = tp_criterion(output['tp_logit'], batch_y)
            timeout_loss = timeout_criterion(output['timeout_pct'], batch_timeout)
            downside_loss = pinball_loss(output['downside_pct'], batch_downside, DOWNSIDE_QUANTILE)
            loss = tp_loss + TIMEOUT_LOSS_WEIGHT * timeout_loss + DOWNSIDE_LOSS_WEIGHT * downside_loss
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            for key, value in [('loss', loss), ('tp', tp_loss), ('timeout', timeout_loss), ('downside', downside_loss)]:
                totals[key] += value.item() * len(batch_x)
        valid_prediction = predict_multitask(model, x_scaled, valid_indices)
        valid_metrics = multitask_metrics(valid_prediction)
        selection_score = valid_metrics['pr_auc'] + 0.25 * valid_metrics['utility_spearman']
        history.append({
            'epoch': epoch, **{f'train_{key}_loss': value / len(train_indices) for key, value in totals.items()},
            **{f'valid_{key}': value for key, value in valid_metrics.items()},
            'selection_score': selection_score, 'learning_rate': optimizer.param_groups[0]['lr'],
        })
        if selection_score > best_score + 1e-4:
            best_score = selection_score
            best_epoch = epoch
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        scheduler.step()
        if stale >= PATIENCE:
            break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), pos_weight_value, best_epoch


fold_rows, histories, oof_frames = [], [], []
final_bundle = None
started = time.time()
for fold_number, valid_session in enumerate(OOF_VALID_SESSIONS, start=1):
    train_sessions = [session for session in sessions if session < valid_session]
    train_indices = np.flatnonzero(metadata['session'].isin(train_sessions).to_numpy())
    valid_indices = np.flatnonzero(metadata['session'].eq(valid_session).to_numpy())
    center, scale = fit_robust_scaler(X, train_indices)
    X_scaled = apply_scaler(X, center, scale)
    periods = discover_periods(X_scaled, train_indices)
    model, history, pos_weight, best_epoch = fit_fold_multitask(
        X_scaled, train_indices, valid_indices, periods, SEED + 100 + fold_number,
    )
    train_prediction = predict_multitask(model, X_scaled, train_indices)
    valid_prediction = predict_multitask(model, X_scaled, valid_indices)
    train_metrics = multitask_metrics(train_prediction)
    valid_metrics = multitask_metrics(valid_prediction)
    fold_rows.append({
        'fold': fold_number, 'valid_session': valid_session, 'train_sessions': train_sessions,
        'periods': periods, 'pos_weight': pos_weight, 'best_epoch': best_epoch,
        **{f'train_{key}': value for key, value in train_metrics.items()},
        **{f'valid_{key}': value for key, value in valid_metrics.items()},
    })
    history['fold'] = fold_number
    history['valid_session'] = valid_session
    histories.append(history)
    frame = metadata.iloc[valid_prediction['index']].copy()
    frame['fold'] = fold_number
    for key in ['tp_probability', 'predicted_timeout', 'predicted_downside']:
        frame[key] = valid_prediction[key]
    oof_frames.append(frame)
    if valid_session == sessions[-2]:
        final_bundle = {
            'state_dict': {name: value.detach().cpu().clone() for name, value in model.state_dict().items()},
            'center': center.copy(), 'scale': scale.copy(), 'periods': periods,
            'train_sessions': train_sessions, 'calibration_session': valid_session,
            'pos_weight': pos_weight, 'best_epoch': best_epoch,
        }
    print(
        f"fold {fold_number} | epoch={best_epoch} | TP AP {train_metrics['pr_auc']:.4f}->{valid_metrics['pr_auc']:.4f} | "
        f"utility rho {train_metrics['utility_spearman']:.3f}->{valid_metrics['utility_spearman']:.3f}"
    )
    del model, X_scaled
    torch.cuda.empty_cache()
    gc.collect()

fold_metrics = pd.DataFrame(fold_rows)
training_history = pd.concat(histories, ignore_index=True)
oof = pd.concat(oof_frames).sort_values('entry_timestamp').reset_index(drop=True)
print(f'elapsed: {(time.time() - started) / 60:.2f} min')
display(fold_metrics[[
    'fold', 'valid_session', 'best_epoch', 'train_pr_auc', 'valid_pr_auc',
    'valid_timeout_spearman', 'valid_downside_spearman', 'valid_utility_spearman',
]])

/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


fold 1 | epoch=8 | TP AP 0.1731->0.1174 | utility rho 0.032->-0.003


/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


fold 2 | epoch=15 | TP AP 0.3560->0.1150 | utility rho 0.176->0.052


/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


fold 3 | epoch=12 | TP AP 0.2648->0.1048 | utility rho 0.117->0.022
elapsed: 2.77 min


,fold,valid_session,best_epoch,train_pr_auc,valid_pr_auc,valid_timeout_spearman,valid_downside_spearman,valid_utility_spearman
0,1,session_2026-07-14,8,0.173065,0.117432,0.201946,0.527581,-0.002917
1,2,session_2026-07-15,15,0.356015,0.115001,0.193645,0.593874,0.051987
2,3,session_2026-07-16,12,0.264813,0.104796,0.198742,0.632725,0.021812


## OOF utility·risk penalty 선택과 최종 Test

OOF에서 downside penalty와 진입 상위 비율을 함께 선택한다. OOF 손익이 양수이고 profit factor가 1보다 클 때만 배포 가능하다. Test threshold는 직전 validation 날짜의 utility 분포로 고정한다.

In [4]:
def cooldown_trade_frame(frame, signal):
    work = frame.copy()
    work['_signal'] = np.asarray(signal, dtype=bool)
    selected_indices = []
    for _, group in work.sort_values('entry_timestamp').groupby(['session', 'symbol'], sort=False):
        available_at = None
        for index, row in group.iterrows():
            if not row['_signal']:
                continue
            if available_at is not None and row['entry_timestamp'] < available_at:
                continue
            selected_indices.append(index)
            hold = int(row['first_hit_minute']) if row['target_net3_within_5m'] == 1 else 5
            available_at = row['entry_timestamp'] + pd.Timedelta(minutes=hold)
    trades = work.loc[selected_indices].sort_values('entry_timestamp').copy()
    trades['trade_return'] = np.where(
        trades['target_net3_within_5m'].eq(1), 0.03, trades['timeout_net_return_5m'],
    )
    trades['pnl_usd'] = STAKE_USD * trades['trade_return']
    return trades


def backtest_metrics(frame, signal):
    trades = cooldown_trade_frame(frame, signal)
    if len(trades) == 0:
        return {'trades': 0, 'sessions_traded': 0, 'precision': np.nan, 'recall': np.nan,
                'win_rate': np.nan, 'mean_return': np.nan, 'median_return': np.nan,
                'risk_adjusted_return': np.nan, 'total_pnl_usd': np.nan,
                'profit_factor': np.nan, 'max_drawdown_usd': np.nan}
    returns = trades['trade_return'].to_numpy()
    pnl = trades['pnl_usd'].to_numpy()
    equity = np.cumsum(pnl)
    peak = np.maximum.accumulate(np.r_[0.0, equity])
    drawdown = np.r_[0.0, equity] - peak
    gross_profit = pnl[pnl > 0].sum()
    gross_loss = -pnl[pnl < 0].sum()
    return {
        'trades': int(len(trades)), 'sessions_traded': int(trades['session'].nunique()),
        'precision': float(trades['target_net3_within_5m'].mean()),
        'recall': float(trades['target_net3_within_5m'].sum() / max(frame['target_net3_within_5m'].sum(), 1)),
        'win_rate': float((returns > 0).mean()), 'mean_return': float(returns.mean()),
        'median_return': float(np.median(returns)),
        'risk_adjusted_return': float(returns.mean() - returns.std(ddof=1) / math.sqrt(len(returns))) if len(returns) > 1 else float(returns.mean()),
        'total_pnl_usd': float(pnl.sum()),
        'profit_factor': float(gross_profit / gross_loss) if gross_loss > 0 else float('inf'),
        'max_drawdown_usd': float(drawdown.min()),
    }


TOP_FRACTIONS = [0.001, 0.0025, 0.005, 0.01, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15, 0.20]
DOWNSIDE_PENALTIES = [0.0, 0.10, 0.25, 0.50, 1.0]
candidate_rows = []
for downside_penalty in DOWNSIDE_PENALTIES:
    score = predicted_utility(
        oof['tp_probability'].to_numpy(), oof['predicted_timeout'].to_numpy(),
        oof['predicted_downside'].to_numpy(), downside_penalty,
    )
    for top_fraction in TOP_FRACTIONS:
        signal = np.zeros(len(oof), dtype=bool)
        thresholds = []
        for fold, group in oof.groupby('fold'):
            fold_score = score[group.index]
            threshold = float(np.quantile(fold_score, 1 - top_fraction))
            thresholds.append(threshold)
            signal[group.index] = fold_score >= threshold
        candidate_rows.append({
            'downside_penalty': downside_penalty, 'top_fraction': top_fraction,
            'mean_fold_threshold': float(np.mean(thresholds)), **backtest_metrics(oof, signal),
        })
candidate_table = pd.DataFrame(candidate_rows)
eligible = candidate_table[
    candidate_table['trades'].ge(MIN_OOF_TRADES) & candidate_table['sessions_traded'].ge(2)
]
if eligible.empty:
    eligible = candidate_table[candidate_table['trades'].gt(0)]
selected = eligible.sort_values(['risk_adjusted_return', 'total_pnl_usd']).iloc[-1]
SELECTED_TOP_FRACTION = float(selected['top_fraction'])
SELECTED_DOWNSIDE_PENALTY = float(selected['downside_penalty'])
DEPLOYMENT_ELIGIBLE = bool(
    selected['risk_adjusted_return'] > 0 and selected['total_pnl_usd'] > 0 and selected['profit_factor'] > 1
)
print('selected penalty/top fraction:', SELECTED_DOWNSIDE_PENALTY, f'{SELECTED_TOP_FRACTION:.2%}')
print('deployment eligible:', DEPLOYMENT_ELIGIBLE)
display(candidate_table.sort_values(['risk_adjusted_return', 'total_pnl_usd'], ascending=False).head(15))

calibration = oof[oof['session'].eq(final_bundle['calibration_session'])].copy()
calibration_score = predicted_utility(
    calibration['tp_probability'].to_numpy(), calibration['predicted_timeout'].to_numpy(),
    calibration['predicted_downside'].to_numpy(), SELECTED_DOWNSIDE_PENALTY,
)
FIXED_THRESHOLD = float(np.quantile(calibration_score, 1 - SELECTED_TOP_FRACTION))
test_indices = np.flatnonzero(metadata['session'].eq(TEST_SESSION).to_numpy())
X_final_scaled = apply_scaler(X, final_bundle['center'], final_bundle['scale'])
final_model = MultiTaskMPTSNet(X.shape[-1], X.shape[1], final_bundle['periods']).to(DEVICE)
final_model.load_state_dict(final_bundle['state_dict'])
test_prediction = predict_multitask(final_model, X_final_scaled, test_indices)
test_predictions = metadata.iloc[test_prediction['index']].copy().reset_index(drop=True)
for key in ['tp_probability', 'predicted_timeout', 'predicted_downside']:
    test_predictions[key] = test_prediction[key]
test_predictions['utility_score'] = predicted_utility(
    test_predictions['tp_probability'].to_numpy(), test_predictions['predicted_timeout'].to_numpy(),
    test_predictions['predicted_downside'].to_numpy(), SELECTED_DOWNSIDE_PENALTY,
)
test_predictions['signal'] = test_predictions['utility_score'].ge(FIXED_THRESHOLD)
test_metrics = multitask_metrics(test_prediction)
test_backtest = backtest_metrics(test_predictions, test_predictions['signal'].to_numpy())
test_trades = cooldown_trade_frame(test_predictions, test_predictions['signal'].to_numpy())
oof_prediction = {
    'index': oof['feature_index'].to_numpy(), 'tp_probability': oof['tp_probability'].to_numpy(),
    'predicted_timeout': oof['predicted_timeout'].to_numpy(),
    'predicted_downside': oof['predicted_downside'].to_numpy(),
}
oof_metrics = multitask_metrics(oof_prediction)

model_path = MODEL_ROOT / f'{MODEL_VERSION}.pt'
metrics_path = RESULT_ROOT / f'{MODEL_VERSION}_metrics.json'
oof_path = RESULT_ROOT / f'{MODEL_VERSION}_oof_predictions.parquet'
test_path = RESULT_ROOT / f'{MODEL_VERSION}_test_predictions.parquet'
trades_path = RESULT_ROOT / f'{MODEL_VERSION}_test_trades.parquet'
candidate_path = RESULT_ROOT / f'{MODEL_VERSION}_threshold_candidates.parquet'
fold_path = RESULT_ROOT / f'{MODEL_VERSION}_fold_metrics.parquet'
history_path = RESULT_ROOT / f'{MODEL_VERSION}_training_history.parquet'
torch.save({
    'model_version': MODEL_VERSION, 'architecture': 'MultiTaskMPTSNet',
    'state_dict': final_bundle['state_dict'], 'input_dim': X.shape[-1], 'seq_len': X.shape[1],
    'periods': final_bundle['periods'], 'feature_names': schema['default_feature_names'],
    'scaler_center': torch.from_numpy(final_bundle['center'].copy()),
    'scaler_scale': torch.from_numpy(final_bundle['scale'].copy()),
    'train_sessions': final_bundle['train_sessions'],
    'calibration_session': final_bundle['calibration_session'], 'test_session': TEST_SESSION,
    'pos_weight': final_bundle['pos_weight'], 'best_epoch': final_bundle['best_epoch'],
    'selected_top_fraction': SELECTED_TOP_FRACTION,
    'selected_downside_penalty': SELECTED_DOWNSIDE_PENALTY,
    'fixed_threshold': FIXED_THRESHOLD, 'deployment_eligible': DEPLOYMENT_ELIGIBLE,
}, model_path)
oof.to_parquet(oof_path, index=False, compression='zstd')
test_predictions.to_parquet(test_path, index=False, compression='zstd')
test_trades.to_parquet(trades_path, index=False, compression='zstd')
candidate_table.to_parquet(candidate_path, index=False, compression='zstd')
fold_metrics.to_parquet(fold_path, index=False, compression='zstd')
training_history.to_parquet(history_path, index=False, compression='zstd')
baseline_metrics = json.loads((RESULT_ROOT / 'compact_mptsnet_weighted_net3_minbid_v1_metrics.json').read_text())
metrics = {
    'model_version': MODEL_VERSION, 'parameters': sum(p.numel() for p in final_model.parameters()),
    'preprocess_version': PREPROCESS_VERSION, 'train_sessions': final_bundle['train_sessions'],
    'calibration_session': final_bundle['calibration_session'], 'test_session': TEST_SESSION,
    'periods': final_bundle['periods'], 'pos_weight': final_bundle['pos_weight'],
    'best_epoch': final_bundle['best_epoch'], 'selected_top_fraction': SELECTED_TOP_FRACTION,
    'selected_downside_penalty': SELECTED_DOWNSIDE_PENALTY, 'fixed_threshold': FIXED_THRESHOLD,
    'deployment_eligible': DEPLOYMENT_ELIGIBLE, 'oof_metrics': oof_metrics,
    'selected_oof_backtest': {key: (None if pd.isna(value) else float(value)) for key, value in selected.to_dict().items()},
    'test_metrics': test_metrics, 'test_backtest': test_backtest,
    'tp_only_baseline_test': {
        'pr_auc': baseline_metrics['test_classification']['pr_auc'],
        **baseline_metrics['test_backtest'],
    },
}
metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
comparison = pd.DataFrame([
    {'model': 'TP-only', 'pr_auc': baseline_metrics['test_classification']['pr_auc'], **baseline_metrics['test_backtest']},
    {'model': 'Multi-task', 'pr_auc': test_metrics['pr_auc'], **test_backtest},
])
print('model:', model_path)
print('safe deployment:', DEPLOYMENT_ELIGIBLE)
display(comparison)
display(pd.Series(test_metrics, name='Test multi-task metrics').to_frame())
display(test_trades[[
    'entry_timestamp', 'symbol', 'tp_probability', 'predicted_timeout', 'predicted_downside',
    'utility_score', 'target_net3_within_5m', 'trade_return', 'pnl_usd',
]])

selected penalty/top fraction: 0.0 0.25%
deployment eligible: False


,downside_penalty,top_fraction,mean_fold_threshold,trades,sessions_traded,precision,recall,win_rate,mean_return,median_return,risk_adjusted_return,total_pnl_usd,profit_factor,max_drawdown_usd
0,0.00,0.0010,0.026983,15,3,0.066667,0.001096,0.733333,0.009565,0.012918,0.002378,14.348176,2.648462,-7.302222
1,0.00,0.0025,0.026583,36,3,0.194444,0.007675,0.583333,0.004051,0.009191,-0.000623,14.582015,1.418463,-17.297647
52,1.00,0.1000,-0.008186,576,3,0.003472,0.002193,0.236111,-0.002726,-0.001877,-0.003154,-157.002300,0.387485,-162.924276
51,1.00,0.0750,-0.007141,461,3,0.002169,0.001096,0.236443,-0.002946,-0.002057,-0.003409,-135.830190,0.350617,-141.752166
50,1.00,0.0500,-0.006222,343,3,0.002915,0.001096,0.250729,-0.002897,-0.001545,-0.003454,-99.353757,0.362780,-105.275733
13,0.10,0.0050,0.021382,60,3,0.133333,0.008772,0.533333,-0.000056,0.003681,-0.003666,-0.335047,0.994991,-21.603964
53,1.00,0.1500,-0.011427,851,3,0.008226,0.007675,0.253819,-0.003376,-0.002264,-0.003770,-287.287045,0.366272,-298.495514
54,1.00,0.2000,-0.014506,1147,3,0.012206,0.015351,0.253705,-0.003862,-0.002315,-0.004277,-442.926159,0.384660,-454.459775
49,1.00,0.0300,-0.005250,228,3,0.000000,0.000000,0.250000,-0.003548,-0.002224,-0.004311,-80.902080,0.344050,-86.824056
33,0.50,0.0010,0.010996,17,3,0.000000,0.000000,0.411765,0.000713,-0.004293,-0.004748,1.212259,1.090458,-8.939638


/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


model: /home/user/urbandatalab/YSLee/model/multitask_mptsnet_tp_timeout_downside_v1.pt
safe deployment: False


,model,pr_auc,trades,sessions_traded,precision,recall,win_rate,mean_return,median_return,risk_adjusted_return,total_pnl_usd,profit_factor,max_drawdown_usd
0,TP-only,0.049531,293,1,0.054608,0.122137,0.307167,-0.014323,-0.009486,-0.016436,-419.676695,0.305545,-424.766599
1,Multi-task,0.046386,6,1,0.000000,0.000000,0.500000,-0.019451,-0.011659,-0.039890,-11.670348,0.335634,-15.955719


,Test multi-task metrics
samples,6609.000000
positives,131.000000
prevalence,0.019821
pr_auc,0.046386
roc_auc,0.728391
brier,0.098064
timeout_mae,0.013003
timeout_spearman,0.272955
downside_mae,0.011971
downside_spearman,0.697251


,entry_timestamp,symbol,tp_probability,predicted_timeout,predicted_downside,utility_score,target_net3_within_5m,trade_return,pnl_usd
6018,2026-07-17 12:46:00+00:00,STAK,0.883318,0.003762,-0.066335,0.026939,0,-0.094303,-9.430295
6032,2026-07-17 13:00:00+00:00,STAK,0.878263,0.004131,-0.061015,0.026851,0,-0.057397,-5.739713
2230,2026-07-17 13:33:00+00:00,CJMB,0.939567,0.000988,-0.056066,0.028247,0,0.016104,1.610427
2235,2026-07-17 13:38:00+00:00,CJMB,0.918487,-0.003859,-0.059658,0.027240,0,-0.023961,-2.396138
2246,2026-07-17 13:49:00+00:00,CJMB,0.923539,0.004795,-0.052231,0.028073,0,0.042211,4.221073
1874,2026-07-17 18:25:00+00:00,BNRG,0.909241,-0.001817,-0.041794,0.027112,0,0.000643,0.064298
